In [1]:
# Cell 1: Installation
!pip install sympy gradio -q
print("✅ Libraries Installed Successfully!")

✅ Libraries Installed Successfully!


In [2]:
# Cell 2: Imports
import sympy as sp
import gradio as gr
import random
import re

# Define mathematical symbols
x, y, z = sp.symbols('x y z')
print("✅ Symbols & Libraries Ready!")

✅ Symbols & Libraries Ready!


In [3]:
# Cell 3: Core Math Logic

def preprocess_input(user_input):
    """Cleans input to make it compatible with SymPy."""
    expr = user_input.strip()
    expr = expr.replace('^', '**')  # Power
    expr = re.sub(r'(\d)([a-zA-Z])', r'\1*\2', expr) # 2x -> 2*x
    if '=' in expr:
        parts = expr.split('=')
        if len(parts) == 2:
            lhs = parts[0].strip()
            rhs = parts[1].strip()
            expr = f"({lhs}) - ({rhs})"
    return expr

def math_tutor_engine(user_problem):
    """Solves the problem and returns step-by-step explanation."""
    if not user_problem:
        return "No input detected."

    try:
        clean_expr = preprocess_input(user_problem)
        problem_type = "simplification"
        steps = []
        result = None

        # Intent Detection
        if "derive" in user_problem.lower() or "differentiate" in user_problem.lower():
            problem_type = "derivative"
        elif "integrate" in user_problem.lower():
            problem_type = "integral"
        elif "=" in user_problem:
            problem_type = "equation"

        # Processing
        if problem_type == "derivative":
            expr_str = clean_expr.replace("derive", "").replace("differentiate", "").strip()
            expr = sp.sympify(expr_str)
            result = sp.diff(expr, x)
            steps.append(f"1️⃣ **Input:** `{user_problem}`")
            steps.append(f"2️⃣ **Operation:** Differentiation")
            steps.append(f"3️⃣ **Process:** Applied derivative rules on `{expr}`.")
            steps.append(f"4️⃣ **Result:** `{result}`")

        elif problem_type == "integral":
            expr_str = clean_expr.replace("integrate", "").strip()
            expr = sp.sympify(expr_str)
            result = sp.integrate(expr, x)
            steps.append(f"1️⃣ **Input:** `{user_problem}`")
            steps.append(f"2️⃣ **Operation:** Integration")
            steps.append(f"3️⃣ **Process:** Integrated `{expr}` with respect to `x`.")
            steps.append(f"4️⃣ **Result:** `{result}` + C")

        elif problem_type == "equation":
            expr = sp.sympify(clean_expr)
            result = sp.solve(expr, x)
            steps.append(f"1️⃣ **Equation:** `{user_problem}`")
            steps.append(f"2️⃣ **Goal:** Find `x`.")
            steps.append(f"3️⃣ **Method:** Isolated variable `x`.")
            steps.append(f"4️⃣ **Solution:** `{result}`")

        else:
            expr = sp.sympify(clean_expr)
            result = sp.simplify(expr)
            steps.append(f"1️⃣ **Input:** `{user_problem}`")
            steps.append(f"2️⃣ **Operation:** Simplification/Calculation")
            steps.append(f"3️⃣ **Result:** `{result}`")

        steps_text = "\n".join(steps)
        return f"✅ **Final Answer:** `{result}`\n\n---\n**📖 Explanation:**\n{steps_text}"

    except Exception as e:
        return f"❌ **Error:** I couldn't understand that. Try `2x+5=10` or `derive x^2`."

In [4]:
# Cell 4: Game Logic

# Global State
state = {
    "score": 0,
    "streak": 0,
    "active_q": None,
    "active_a": None
}

def generate_challenge():
    """Generates random Linear or Quadratic equations."""
    a = random.randint(1, 10)
    b = random.randint(1, 10)
    if random.random() > 0.5:
        problem = f"{a}*x + {b} = 0"
        solution = sp.solve(sp.sympify(f"{a}*x + {b}"), x)
    else:
        problem = f"x**2 - {a} = 0"
        solution = sp.solve(sp.sympify(f"x**2 - {a}"), x)
    return problem, solution

def ask_challenge():
    problem, solution = generate_challenge()
    state["active_q"] = problem
    state["active_a"] = str(solution)
    return f"🎯 **Challenge:**\n\n{problem}\n\nSolve for `x`!"

def check_game_answer(user_text):
    user_text = str(user_text).strip()
    correct_answer = state["active_a"].strip()

    if user_text in correct_answer or correct_answer in user_text:
        state["score"] += 10
        state["streak"] += 1
        return f"✅ **CORRECT!**\n\n🏆 **Score:** {state['score']}\n🔥 **Streak:** {state['streak']}"
    else:
        explanation = math_tutor_engine(state["active_q"])
        state["streak"] = 0
        return f"❌ **Wrong!** Correct was: {correct_answer}\n\n{explanation}"

In [5]:
# Cell 5: Interface Helper Functions

def interact(message, history):
    """Handles the chat interaction logic."""
    # Check if we are in Game Mode
    if state["active_q"] is not None:
        response = check_game_answer(message)
        state["active_q"] = None # Reset game mode
    else:
        # Normal Math Mode
        response = math_tutor_engine(message)

    # Update Chat History (New Format)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return history, f"🏆 Score: {state['score']}", f"🔥 Streak: {state['streak']}"

def switch_theme(theme_name):
    """Switches the app theme."""
    if "Midnight" in theme_name:
        return gr.themes.Soft(primary_hue="slate", neutral_hue="slate").set(body_background_fill="#0f172a")
    elif "Cyberpunk" in theme_name:
        return gr.themes.Soft(primary_hue="pink", secondary_hue="purple").set(body_background_fill="#1e1b4b")
    else:
        return gr.themes.Soft()

In [ ]:
# Cell 6: UI Design and Launch

with gr.Blocks(theme=gr.themes.Soft()) as app:
    with gr.Row():
        # Sidebar
        with gr.Column(scale=1, min_width=250):
            gr.Markdown("### ⚙️ Settings")
            theme_drop = gr.Dropdown(["Ocean (Light)", "Midnight (Dark)", "Cyberpunk"], label="Change Theme", value="Ocean (Light)")

            gr.Markdown("---")
            gr.Markdown("### 🎮 Game Zone")
            game_btn = gr.Button("⚡ Challenge Me", variant="primary")
            score_display = gr.Markdown("🏆 Score: 0")
            streak_display = gr.Markdown("🔥 Streak: 0")

        # Main Chat
        with gr.Column(scale=4):
            gr.Markdown("<h1 style='text-align: center;'>🧮 Math Genius AI</h1>")

            # Chatbot with modern settings
            chatbot = gr.Chatbot(
                height=500,
                show_label=False,
                type="messages",
                allow_tags=True,
                show_copy_button=True
            )

            with gr.Row():
                txt = gr.Textbox(show_label=False, placeholder="Type math or answer...", scale=9)
                btn = gr.Button("Send", variant="primary", scale=1)

    # Connections
    btn.click(interact, [txt, chatbot], [chatbot, score_display, streak_display])
    txt.submit(interact, [txt, chatbot], [chatbot, score_display, streak_display])
    game_btn.click(ask_challenge, outputs=txt)
    theme_drop.change(switch_theme, inputs=[theme_drop], outputs=[app])

# Launch
app.launch(share=True, debug=True)

/tmp/ipykernel_324/1463842287.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:
/tmp/ipykernel_324/1463842287.py:21: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d15275eaf79391d459.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
